# RLHF, DPO, and GRPO — interactive companion

Companion to [Post 2d: RLHF, DPO, and GRPO](../posts/02d-rlhf-dpo-grpo.qmd).
The full bridge from policy gradients to LLM post-training, on a tiny
synthetic task you can step through.

**What you'll do (≈ 25 minutes):**
1. Build a synthetic preference task with known true rewards.
2. Fit a reward model from preference pairs via Bradley-Terry.
3. Train DPO directly on preferences — no reward model needed.
4. Train GRPO with a group-relative baseline (the DeepSeek-R1 recipe).
5. See how a corrupted reward model causes reward hacking, and what KL does about it.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.policy_gradient import SoftmaxPolicy
from nano_agents.rlhf import (
    PreferenceTask, TabularRewardModel,
    train_dpo, train_grpo,
)

## 1. Bradley-Terry preferences

For two completions $y_a, y_b$ of a prompt $x$ with hidden rewards
$r_a, r_b$, the Bradley-Terry model says

$$
P(y_a \succ y_b \mid x) = \sigma(r_a - r_b).
$$

A reward gap of 2 gives 88% preference probability; a gap of 0 gives 50%.
Visualize it:

In [ ]:
diff = np.linspace(-5, 5, 500)
p = 1.0 / (1.0 + np.exp(-diff))
plt.plot(diff, p, color="C0", linewidth=2)
plt.fill_between(diff, p, alpha=0.18)
plt.axhline(0.5, color="black", linestyle=":", alpha=0.5)
plt.axvline(0.0, color="black", linestyle=":", alpha=0.5)
for d in [-2, 0, 2]:
    pp = 1.0 / (1.0 + np.exp(-d))
    plt.scatter([d], [pp], s=80, color="C3", zorder=5)
    plt.annotate(f"$\\Delta r$={d}\n$P$={pp:.2f}", (d, pp),
                 xytext=(10, -18 if pp > 0.5 else 10),
                 textcoords="offset points", fontsize=9)
plt.xlabel(r"reward difference $\Delta r$"); plt.ylabel(r"$P(y_w \succ y_l)$")
plt.title("Bradley-Terry preference model"); plt.grid(alpha=0.3); plt.show()

## 2. Build a synthetic preference task

`PreferenceTask` generates a small contextual problem (4 prompts × 4 completions)
with hidden true rewards. We can sample noisy Bradley-Terry preferences and
test whether our algorithms recover the right policy.

In [ ]:
task = PreferenceTask(n_contexts=4, n_completions=4, reward_scale=2.0, seed=0)
print("True rewards (rows = contexts, cols = completions):")
print(np.round(task.true_rewards, 2))
print(f"\\nGreedy policy: completion {task.greedy_policy()} per context")
opt = task.expected_reward_under_policy(np.eye(4)[task.greedy_policy()])
print(f"Optimal expected reward: {opt:.3f}")

## 3. Reward model from preferences

Fit a tabular reward model by maximizing the Bradley-Terry likelihood.
With enough preferences, the learned reward recovers the true reward
*up to per-context additive constants* (which the model can't identify).

In [ ]:
rng = np.random.default_rng(0)
prefs = task.collect_preferences(n_samples=2000, rng=rng)

rm = TabularRewardModel(task.n_contexts, task.n_completions)
losses = rm.fit(prefs, n_steps=600, lr=0.5)

# Center per context to remove the additive ambiguity.
true_centered = task.true_rewards - task.true_rewards.mean(1, keepdims=True)
learned_centered = rm.r - rm.r.mean(1, keepdims=True)

plt.scatter(true_centered.flatten(), learned_centered.flatten(),
             s=70, alpha=0.75, color="C0", edgecolor="white", linewidth=0.8)
lo = min(true_centered.min(), learned_centered.min()) - 0.2
hi = max(true_centered.max(), learned_centered.max()) + 0.2
plt.plot([lo, hi], [lo, hi], "k--", alpha=0.5, label="perfect recovery")
plt.xlabel("true reward (centered)"); plt.ylabel("learned reward (centered)")
plt.title(f"Reward model recovery from {len(prefs)} preferences")
plt.legend(); plt.grid(alpha=0.3); plt.show()

Points cluster near the identity line — the reward model is recovering the
ordering within each context.

### Try this
- Drop to `n_samples=50`. The scatter loosens; the ordering is less reliable.
- Set `n_samples=20`. Some contexts may have their argmax wrong.

## 4. DPO — direct preference optimization

The DPO loss skips the reward model entirely:

$$
\mathcal{L}_{\rm DPO} = -\log\sigma\!\left[\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\rm ref}(y_w|x)}
- \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\rm ref}(y_l|x)}\right].
$$

It's just gradient descent on a binary classification objective. No rollouts,
no critic, no PPO loop.

In [ ]:
ref = SoftmaxPolicy(task.n_contexts, task.n_completions)   # uniform reference
policy = SoftmaxPolicy(task.n_contexts, task.n_completions)
losses = train_dpo(policy, ref, prefs, n_steps=1500, lr=1.0, beta=0.1)

# Evaluate.
probs = np.array([policy.probs(c) for c in range(task.n_contexts)])
dpo_return = task.expected_reward_under_policy(probs)
print(f"DPO final expected reward: {dpo_return:.3f}  (optimum = {opt:.3f})")

# Plot the DPO loss decay.
plt.semilogy(losses)
plt.xlabel("DPO step"); plt.ylabel("loss (log scale)")
plt.title("DPO loss decays roughly logarithmically — needs many steps")
plt.grid(which="both", alpha=0.3); plt.show()

DPO reaches close to optimal expected reward without ever computing a reward.
The loss decays logarithmically — getting the last few percent requires
many more steps because the loss is `softplus(-margin)`.

### Try this
- Set `beta=0.05` (weaker constraint). With less pull on the margin, DPO
  needs many *more* steps. Does it eventually reach optimal?
- Set `beta=1.0`. Stronger gradient but the optimization becomes unstable —
  what do you observe?
- Use only 100 preferences. The result should degrade gracefully.

## 5. GRPO — group-relative baseline

GRPO is PPO without a learned critic. For each prompt, sample G completions
and use their *mean reward* as the baseline. The clipped surrogate from
PPO is unchanged.

In [ ]:
# Use the learned reward model as the reward signal (RLHF pipeline).
policy = SoftmaxPolicy(task.n_contexts, task.n_completions)
hist = train_grpo(task, policy,
                   n_iterations=300, group_size=8,
                   n_prompts_per_iter=2, n_epochs=4,
                   lr=0.15, eps=0.2,
                   use_true_reward=False, reward_model=rm,
                   rng=np.random.default_rng(0))

# Same evaluation.
probs = np.array([policy.probs(c) for c in range(task.n_contexts)])
grpo_return = task.expected_reward_under_policy(probs)
print(f"GRPO+RM final expected reward: {grpo_return:.3f}  (optimum = {opt:.3f})")

plt.plot(hist["expected_return"])
plt.axhline(opt, color="black", linestyle="--", alpha=0.5, label="optimum")
plt.xlabel("iteration"); plt.ylabel("expected TRUE reward")
plt.title("GRPO learning curve (training on learned reward model)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

GRPO with a well-fit reward model also reaches optimal. No critic needed —
the within-prompt group mean handles the variance reduction.

### Try this
- Set `use_true_reward=True` and re-run. Should reach optimum even faster
  (no reward-model error to navigate).
- Set `group_size=2` (minimal group). Does the group baseline still work?
- Set `n_epochs=1` (single update per batch). Slower, but does it converge?

## 6. Reward hacking and the KL penalty

What happens if the reward model is *wrong*? Let's inject a deliberate bias
that says one specific completion is much better than it really is.

In [ ]:
class CorruptedRewardModel:
    def __init__(self, true_rewards, c_hack, a_hack, bias):
        self.r = true_rewards.copy()
        self.r[c_hack, a_hack] += bias
    def predict(self, c, a):
        return float(self.r[c, a])

# Add a +5 bonus to a completion that is NOT actually best.
true_best = task.true_rewards.argmax(axis=1)
c_hack = 0
a_hack = next(a for a in range(4) if a != true_best[c_hack])
print(f"Corrupting context {c_hack}: pretending completion {a_hack} (real reward "
      f"{task.true_rewards[c_hack, a_hack]:.2f}) is the best.")
corrupted = CorruptedRewardModel(task.true_rewards, c_hack, a_hack, bias=5.0)

# Train with different KL penalty strengths.
ref = SoftmaxPolicy(task.n_contexts, task.n_completions)
results = {}
for kl in [0.0, 1.0, 5.0]:
    policy = SoftmaxPolicy(task.n_contexts, task.n_completions)
    hist = train_grpo(task, policy,
                       n_iterations=200, group_size=8,
                       n_prompts_per_iter=2, n_epochs=4,
                       lr=0.15, eps=0.2,
                       use_true_reward=False, reward_model=corrupted,
                       track_kl_against_ref=True, ref_policy=ref, kl_coef=kl,
                       rng=np.random.default_rng(0))
    results[kl] = hist

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for kl, hist in results.items():
    axes[0].plot(hist["expected_return"], label=rf"$\beta_{{\rm KL}}={kl}$")
    axes[1].plot(hist["kl_to_ref"], label=rf"$\beta_{{\rm KL}}={kl}$")
axes[0].axhline(opt, color="black", linestyle="--", alpha=0.5, label="true optimum")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("expected TRUE reward")
axes[0].set_title("True reward under corrupted RM"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("iteration"); axes[1].set_ylabel(r"KL$(\pi \| \pi_{\rm ref})$")
axes[1].set_title("Policy drift from reference"); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Honest finding** (also discussed in Post 2d §4.3): the KL penalty does
**not** cleanly fix reward hacking.

- $\beta_{\rm KL} = 0$: policy drifts freely (KL → 1.3), achieves moderate
  true reward (~0.95) by chasing the corrupted RM.
- $\beta_{\rm KL} = 1$: similar story, slight reduction in drift.
- $\beta_{\rm KL} = 5$: KL stays near zero, but the policy can't learn —
  true reward stuck at ~0.35.

There's no $\beta_{\rm KL}$ that simultaneously prevents reward hacking
*and* allows useful learning. Real fixes for reward hacking come from
elsewhere — reward model ensembles, calibrated uncertainty, constitutional
constraints. The KL penalty is necessary but not sufficient.

### Try this
- Increase the corruption bias to 20. Does the picture change qualitatively?
- Corrupt *all* contexts simultaneously. Now KL is fighting a much stronger
  signal — does it ever help?

## What's next

This closes Topic 2. You've built the entire policy-gradient stack
from first principles:

1. **Log-derivative trick** → REINFORCE (Post 2a).
2. **Baselines** → A2C with GAE (Post 2b).
3. **Trust regions** → PPO (Post 2c).
4. **Preferences** → RLHF / DPO / GRPO (Post 2d).

Every modern RLHF system is some variation within this design space.
DPO is PPO with the reward model algebraically eliminated. GRPO is PPO
with the critic replaced by a group baseline. The fundamentals are what
you just implemented.

**Topic 3** moves from training a policy to *using* one intelligently:
chain-of-thought, tool use, planning. The first notebook will be
`03a-inference-time-reasoning.ipynb` once that post is written.